In [0]:
spark.conf.set("fs.azure.account.auth.type.proddatalakefordatabrick.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.proddatalakefordatabrick.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.proddatalakefordatabrick.dfs.core.windows.net", "8b03b30e-c838-4568-a41e-a801634f0e96")
spark.conf.set("fs.azure.account.oauth2.client.secret.proddatalakefordatabrick.dfs.core.windows.net", dbutils.secrets.get(scope="darshanscope", key="app-secret"))
spark.conf.set("fs.azure.account.oauth2.client.endpoint.proddatalakefordatabrick.dfs.core.windows.net", "https://login.microsoftonline.com/5ff33f54-37cd-4d6d-9156-f6c00f164999/oauth2/token")

In [0]:
dbutils.widgets.text("file_name", "")
p_file_name=dbutils.widgets.get("file_name")

In [0]:
df=spark.readStream.format('cloudFiles').option('cloudFiles.format','parquet').option('cloudFiles.schemaLocation',f'abfss://destination@proddatalakefordatabrick.dfs.core.windows.net/bronze/checkpoint_{p_file_name}').load(f'abfss://source@proddatalakefordatabrick.dfs.core.windows.net/{p_file_name}')

In [0]:
df.writeStream.format('parquet').outputMode('append').option('checkpointLocation',f'abfss://destination@proddatalakefordatabrick.dfs.core.windows.net/bronze/checkpoint_{p_file_name}').option('path',f'abfss://destination@proddatalakefordatabrick.dfs.core.windows.net/bronze/{p_file_name}').trigger(once=True).start()

In [0]:
bronze_df = spark.read.format('parquet').load(f"abfss://destination@proddatalakefordatabrick.dfs.core.windows.net/bronze/{p_file_name}")
bronze_df.count()
#Final Comit - Bronze
